In [1]:
from functools import cache
from pathlib import Path

import pandas as pd
import google.auth

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.sql import get_engine

from shared_utils import bq_utils

from update_vars import GTFS_DATA_DICT, file_name

# Initialize credentials and DB engine
credentials, project = google.auth.default()
db_engine = get_engine()


In [2]:
@cache
def gcs_pandas():
    return GCSPandas()

In [3]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [4]:
FILEPATH_URL = "gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/crosswalk_2026_06.parquet"

In [5]:
 f"{GTFS_DATA_DICT.gcs_paths.DIGEST_GCS}processed/{GTFS_DATA_DICT.gtfs_digest_rollup.ntd_profile}_{file_name}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/ntd_profile_2026_06.parquet'

In [6]:
f"{GTFS_DATA_DICT.gcs_paths.DIGEST_GCS}processed/{GTFS_DATA_DICT.gtfs_digest_rollup.crosswalk}_{file_name}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/crosswalk_2026_06.parquet'

In [7]:
df = (
        gcs_pandas()
        .read_parquet(FILEPATH_URL)
    )

In [8]:
df.head()

,name,analysis_name,county_name,caltrans_district,caltrans_district_int,ntd_id,ntd_id_2022
0,Delano Schedule,City of Delano,Kern,06-Fresno / Bakersfield,6,90238,90238
1,OmniTrans Schedule,OmniTrans,San Bernardino,08-San Bernardino / Riverside,8,90029,90029
2,SLO Schedule,San Luis Obispo Regional Transit Authority,San Luis Obispo,05-San Luis Obispo / Santa Barbara,5,90156,90156
3,San Joaquin Schedule,San Joaquin Regional Transit District,San Joaquin,10-Stockton,10,90012,90012
4,Tuolumne Remix Schedule,Tuolumne County Transit Agency,Tuolumne,10-Stockton,10,9R02-91057,91057


In [9]:
df.sort_values(by = ["analysis_name"]).head(15)

,name,analysis_name,county_name,caltrans_district,caltrans_district_int,ntd_id,ntd_id_2022
49,Bay Area 511 AC Transit Schedule,Alameda-Contra Costa Transit District,Alameda,04-Bay Area / Oakland,4,90014,90014
140,Amador Schedule,Amador Regional Transit System,Amador,10-Stockton,10,9R02-91000,91000
31,Antelope Valley Transit Authority Schedule,Antelope Valley Transit Authority,Los Angeles,07-Los Angeles / Ventura,7,90121,90121
104,B-Line Schedule,Butte County Association of Governments,Butte,03-Marysville / Sacramento,3,90208,90208
158,Calaveras Schedule,Calaveras Transit Agency,Calaveras,10-Stockton,10,9R02-99442,99442
157,Bay Area 511 County Connection Schedule,Central Contra Costa Transit Authority,Contra Costa,04-Bay Area / Oakland,4,90078,90078
77,Havasu Landing Ferry Schedule,Chemehuevi Indian Tribe,San Bernardino,08-San Bernardino / Riverside,8,99316,99316
88,Bay Area 511 Golden Gate Park Shuttle Schedule,City and County of San Francisco,San Francisco,04-Bay Area / Oakland,4,90015,90015
75,Bay Area 511 Muni Schedule,City and County of San Francisco,San Francisco,04-Bay Area / Oakland,4,90015,90015
8,Golden Gate Park Shuttle Schedule,City and County of San Francisco,San Francisco,04-Bay Area / Oakland,4,90015,90015


In [10]:
df.loc[df.analysis_name.str.contains("Banning")]

,name,analysis_name,county_name,caltrans_district,caltrans_district_int,ntd_id,ntd_id_2022


In [11]:
df.loc[df.name.str.contains("Banning")]

,name,analysis_name,county_name,caltrans_district,caltrans_district_int,ntd_id,ntd_id_2022


## `_prep_crosswalk_ntd.py`

In [12]:
og_df = bq_utils.download_table(
        project_name="cal-itp-data-infra",
        dataset_name="mart_transit_database",
        table_name="bridge_gtfs_analysis_name_x_ntd",
        date_col=None,
    )

Downloading: 100%|██████████|
query: SELECT * FROM  `cal-itp-data-infra`.`mart_transit_database`.`bridge_gtfs_analysis_name_x_ntd`


In [13]:
og_df.head()

,organization_name,organization_source_record_id,schedule_source_record_id,schedule_gtfs_dataset_name,analysis_name,regional_feed_type,county_name,caltrans_district,caltrans_district_name,caltrans_district_full,ntd_id,ntd_id_2022,rtpa_name,mpo_name
0,Amtrak,recKsb5FnJy70up78,recIHiLOHYXfVknaq,Amtrak Schedule,Amtrak,None,Sacramento,3,Marysville / Sacramento,03 - Marysville / Sacramento,None,None,Sacramento Area Council of Governments,Sacramento Area Council of Governments
1,San Joaquin Regional Rail Commission,recpgYVeU3VePMeWx,recdl7HAeF4XOqGQM,Bay Area 511 ACE Schedule,San Joaquin Regional Rail Commission,Regional Subfeed,San Joaquin,10,Stockton,10 - Stockton,90182,90182,San Joaquin Council of Governments,San Joaquin Council of Governments
2,City and County of San Francisco,rechaapWbeffO33OX,recHD22phgJs34JHP,Bay Area 511 Muni Schedule,City and County of San Francisco,Regional Subfeed,San Francisco,4,Bay Area / Oakland,04 - Bay Area / Oakland,90015,90015,Metropolitan Transportation Commission,Metropolitan Transportation Commission
3,Glenn County,rec0sMQyK2v8Cs4Io,reclMnN6OEOAa0mRU,Glenn Schedule,Glenn County,None,Glenn,3,Marysville / Sacramento,03 - Marysville / Sacramento,9R02-91088,91088,Glenn County Transportation Commission,None
4,Humboldt Transit Authority,recaa3naoNR4a5RsJ,recrGgXZxqm3dOPH5,Humboldt Flex,Humboldt Transit Authority,None,Humboldt,1,Eureka,01 - Eureka,9R02-91036,91036,Humboldt County Association of Governments,None


In [14]:
og_df.loc[og_df.organization_name.str.contains("Banning")]

,organization_name,organization_source_record_id,schedule_source_record_id,schedule_gtfs_dataset_name,analysis_name,regional_feed_type,county_name,caltrans_district,caltrans_district_name,caltrans_district_full,ntd_id,ntd_id_2022,rtpa_name,mpo_name
117,City of Banning,recuGkFhN2WXGK67H,recnAiZYHWBxUwH0F,Banning Pass Schedule,City of Banning,None,Riverside,8,San Bernardino / Riverside,08 - San Bernardino / Riverside,None,None,Southern California Association of Governments,Southern California Association of Governments


In [18]:
og_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 222 entries, 0 to 221
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   organization_name              222 non-null    object
 1   organization_source_record_id  222 non-null    object
 2   schedule_source_record_id      222 non-null    object
 3   schedule_gtfs_dataset_name     222 non-null    object
 4   analysis_name                  222 non-null    object
 5   regional_feed_type             47 non-null     object
 6   county_name                    222 non-null    object
 7   caltrans_district              222 non-null    Int64 
 8   caltrans_district_name         222 non-null    object
 9   caltrans_district_full         222 non-null    object
 10  ntd_id                         175 non-null    object
 11  ntd_id_2022                    178 non-null    object
 12  rtpa_name                      215 non-null    object
 13  mpo_n

In [15]:
og_df2 = (
        og_df.dropna(subset=["ntd_id", "ntd_id_2022"])
        .drop_duplicates(
            subset=["analysis_name", "organization_name", "schedule_gtfs_dataset_name"]
        )
        .reset_index()
    )

In [17]:
og_df2.loc[og_df2.organization_name.str.contains("Banning")]

,index,organization_name,organization_source_record_id,schedule_source_record_id,schedule_gtfs_dataset_name,analysis_name,regional_feed_type,county_name,caltrans_district,caltrans_district_name,caltrans_district_full,ntd_id,ntd_id_2022,rtpa_name,mpo_name


In [19]:
og_df3 = (
        og_df
        .drop_duplicates(
            subset=["analysis_name", "organization_name", "schedule_gtfs_dataset_name"]
        )
        .reset_index()
    )

In [20]:
og_df3.loc[og_df3.organization_name.str.contains("Banning")]

,index,organization_name,organization_source_record_id,schedule_source_record_id,schedule_gtfs_dataset_name,analysis_name,regional_feed_type,county_name,caltrans_district,caltrans_district_name,caltrans_district_full,ntd_id,ntd_id_2022,rtpa_name,mpo_name
117,117,City of Banning,recuGkFhN2WXGK67H,recnAiZYHWBxUwH0F,Banning Pass Schedule,City of Banning,None,Riverside,8,San Bernardino / Riverside,08 - San Bernardino / Riverside,None,None,Southern California Association of Governments,Southern California Association of Governments


## YML

In [40]:
FILEPATH_URL = f"{GTFS_DATA_DICT.gcs_paths.DIGEST_GCS}processed/{GTFS_DATA_DICT.gtfs_digest_rollup.crosswalk}_{file_name}.parquet"

In [41]:
FILEPATH_URL

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/crosswalk_2026_06.parquet'

In [42]:
crosswalk_og_df = gcs_pandas().read_parquet(FILEPATH_URL)

In [43]:
crosswalk_og_df.head(1)

,name,analysis_name,county_name,caltrans_district,caltrans_district_int,ntd_id,ntd_id_2022
0,Amtrak Schedule,Amtrak,Sacramento,03-Marysville / Sacramento,3,None,None


In [44]:
crosswalk_og_df.loc[crosswalk_og_df.analysis_name.str.contains("Banning")]

,name,analysis_name,county_name,caltrans_district,caltrans_district_int,ntd_id,ntd_id_2022
117,Banning Pass Schedule,City of Banning,Riverside,08-San Bernardino / Riverside,8,None,None


In [45]:
yml_df = (
        gcs_pandas()
        .read_parquet(FILEPATH_URL, columns=["caltrans_district", "analysis_name"])
        .dropna(subset=["caltrans_district"])
        .sort_values(["caltrans_district", "analysis_name"])
        .reset_index(drop=True)
        .drop_duplicates()
    )


In [46]:
yml_df.loc[yml_df.analysis_name.str.contains("Banning")]

,caltrans_district,analysis_name
169,08-San Bernardino / Riverside,City of Banning
